# THE CHSH GAME

We are going to use (a simplified version) of the CHSH game to 'test' whether IBM's quantum computers are really quantum 

In [1]:
%matplotlib inline

import numpy as np
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService

We define a function that creates the circuit that simulates Alice's and Bob's behaviour in the CHSH game. Parameters a0 and a1 are the measurement angles used by Alice when she receives 0 and 1, respectively. For Bob, b0 and b1 play the same role. The default parameters are the ones that give the maximum violation of the corresponding Bell inequality (or, equivalently, the highest winning probability).

In [13]:
#Function to create the circuits for the CHSH game
def CHSH_circuit(x,y,a0=0,a1=np.pi/2,b0=np.pi/4,b1=-np.pi/4):
    
    #x: bit received by Alice
    #y: bit received by Bob
    #a0: measure angle used by Alice when she receives 0
    #a1: measure angle used by Alice when she receives 1
    #b0: measure angle used by Bob when he receives 0
    #b1: measure angle used by Bob when he receives 1
    
    circ = QuantumCircuit(2) 
    
    # First, we create a Bell pair
    
    circ.h(0)
    circ.cx(0,1)

    # Now, we apply rotations for Alice and Bob depending on the bits they have received
    
    if(x==0):
        circ.ry(a0,0)
    else:
        circ.ry(a1,0)

    if(y==0):
        circ.ry(b0,1)
    else:
        circ.ry(b1,1)

    # We measure
        
    circ.measure_all() # Medimos

    return circ
    

We also define a function to compute the winning probability. Notice how we can create a batch of circuits and send them for execution all at the same time.

In [16]:
from qiskit.primitives import StatevectorSampler

def winning_probability(backend, shots = 8192, a0=0,a1=np.pi/2,b0=np.pi/4,b1=-np.pi/4):

    circuits = [CHSH_circuit(0,0,a0,a1,b0,b1), CHSH_circuit(0,1,a0,a1,b0,b1), CHSH_circuit(1,0,a0,a1,b0,b1), CHSH_circuit(1,1,a0,a1,b0,b1)] # We 'pack' four different circuits for execution

    # Run all four circuits
    sampler = StatevectorSampler()
    job = sampler.run(circuits, shots=shots)
    results = job.result()
    
    total = 0
    # For the first three circuits, the winning condition is that Alice's and Bob's outputs are equal
    for i in range(3):
        counts = results[i].data.meas.get_counts()

        total += counts.get("00", 0)
        total += counts.get("11", 0)

    # For the fourth circuit, Alice's and Bob's outputs must be different for them to win
    counts = results[3].data.meas.get_counts()

    total += counts.get("01", 0)
    total += counts.get("10", 0)

        
    return total/(4*shots)    

# First, we try it on the simulator
sampler = StatevectorSampler()

print(winning_probability(sampler))

0.85595703125


We now run on an actual quantum computer. Is Nature classical or quantum? Let's find out! 

In [17]:
# Load the account
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    SamplerV2 as Sampler
)



In [22]:
# Select the least busy real quantum computer
service = QiskitRuntimeService()

backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=2
)

sampler = Sampler(mode=backend)
print("We are executing on...", backend)
print(winning_probability(sampler))


qiskit_runtime_service.__init__:WARNING:2026-08-26 09:30:27,044: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-08-26 09:30:27,596: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-08-26 09:30:34,956: Using instance: open-instance, plan: open


We are executing on... <IBMBackend('ibm_marrakesh')>
0.857086181640625
